# E6 - Associative recall theo do dai chuoi

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

## Cau hoi

Bo loc suy tu corpus co do dai hieu dung **trung vi 2,9 token**. Cau hinh do co the giam perplexity trong khi da vut bo kha nang tam xa - va perplexity khong lo ra dieu do. Tac vu associative recall dat cau tra loi o xa nen no lo ra ngay.

## Lan chay truoc (2026-08-08): mot ket qua dung, mot ket qua vo dung

**Dung - quet chan doan da tra loi duoc cau hoi R5:**

| vocab | L | lop | buoc | do chinh xac | doan mo |
|---:|---:|---|---:|---:|---:|
| 10 | 33 | AA | 744 | 0,194 | 0,100 |
| 10 | 33 | AA | **18720** | **1,000** | 0,100 |
| 10 | 33 | AAA | 18720 | 1,000 | 0,100 |
| 20 | 65 | AAA | 37500 | 0,104 | 0,050 |

=> Tac vu KHONG hong. R5 that bai truoc day chi vi ngan sach huan luyen qua nho (744 buoc). Voi 18720 buoc, attention dat 100%.

**Vo dung - phan so sanh chinh:** o notebook cu, dong chon cau hinh viet `solvable[-1]` (phan tu cuoi danh sach) trong khi comment ghi "cau hinh kho nhat". Phan tu cuoi lai tinh co la cau hinh **de nhat** (vocab=5, L=17). Ket qua: ca bon cau hinh deu dat ~1,000, **tran bao hoa, khong phan biet duoc gi**.

| cau hinh | trung binh | do lech |
|---|---:|---:|
| attention | 1,0000 | 0,0000 |
| hyena_uniform | 0,9995 | 0,0007 |
| hyena_corpus | 0,9995 | 0,0007 |
| hyena_logspace | 0,9990 | 0,0014 |

Chenh lech 0,0005 = 1 mau tren 2000 = nhieu. **Khong duoc trich bang nay lam ket luan**, ngoai mot phat bieu yeu: o do dai 17 token, bo loc corpus khong lam hong recall.

## Thiet ke moi

Doi bien khao sat sang **DO DAI CHUOI**, va bat `unique_keys=True`.

Ly do bat `unique_keys`: khi lay khoa co hoan lai tu bang chu cai nho, chuoi cang DAI thi moi khoa cang xuat hien NHIEU LAN, nen ban sao gan nhat cua khoa truy van cang GAN. Tang do dai chuoi lai lam GIAM khoang cach can nho - do sai dai luong. Voi `unique_keys`, moi khoa xuat hien dung mot lan nen khoang cach trai deu tren toan chuoi (da kiem chung: trung binh = P+1 dung nhu ly thuyet).

Chay hai pha de khoi dot GPU vo ich:
1. **Do tham** - chi attention, tim do dai lon nhat ma tac vu con giai duoc.
2. **So sanh day du** - 4 cau hinh, chi chay o do dai do.

## Cai dat
Settings: **Accelerator = GPU T4 x2**, **Internet = On**.

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK = "/kaggle/working/Hyena-Attention-Study"

import os, shutil, subprocess, sys

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} - {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHUA BAT GPU - Settings > Accelerator > GPU T4 x2")
    print("!" * 70)

In [ ]:
!python tests/test_recall.py

## 2. Ham huan luyen dung chung

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from hyena_study.data.synthetic import (
    RecallConfig, build_recall_dataset, chance_accuracy, query_distance)
from hyena_study.models import HyenaFilterConfig, LMConfig, SequenceLM

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STEPS_TARGET = 18720   # ngan sach da duoc chung minh du o L=33 (xem bang tren)


def run_one(seq_len, layers, alpha=None, seed=0, d_model=64,
            n_train=20000, lr=1e-3, bs=64):
    """Huan luyen mot cau hinh, tra ve (accuracy, chance, acc theo khoang cach)."""
    torch.manual_seed(seed); np.random.seed(seed)
    P = (seq_len - 1) // 2
    cfg = RecallConfig(vocab_size=P, seq_len=seq_len, n_train=n_train,
                       n_val=10, n_test=2000, seed=0, unique_keys=True)
    d = build_recall_dataset(cfg)
    xtr, ytr = d["train"]; xte, yte = d["test"]
    L = xtr.shape[1]

    m = SequenceLM(LMConfig(
        vocab_size=cfg.vocab_size, d_model=d_model, layer_spec=layers,
        max_seq_len=L, dropout=0.0, n_heads=4,
        hyena_filter=HyenaFilterConfig(alpha_values=alpha),
    )).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)

    xtr_t = torch.from_numpy(xtr).to(DEV); ytr_t = torch.from_numpy(ytr).to(DEV)
    per_epoch = len(xtr) // bs
    epochs = max(1, round(STEPS_TARGET / per_epoch))
    total, step, t0 = epochs * per_epoch, 0, time.time()

    for _ in range(epochs):
        perm = torch.randperm(len(xtr), device=DEV)
        for i in range(0, len(xtr) - bs + 1, bs):
            idx = perm[i:i + bs]
            for g in opt.param_groups:
                g["lr"] = lr * min(1.0, (step + 1) / max(total * 0.05, 1))
            loss = F.cross_entropy(m(xtr_t[idx])[:, -1, :], ytr_t[idx])
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            step += 1

    m.eval()
    with torch.no_grad():
        pred = m(torch.from_numpy(xte).to(DEV))[:, -1, :].argmax(-1).cpu().numpy()
    acc = float((pred == yte).mean())

    # do chinh xac theo khoang cach: do chinh xac tong the co the che giau viec
    # mo hinh chi giai duoc cac cap o gan
    dist = query_distance(xte)
    half = np.median(dist)
    near = float((pred[dist <= half] == yte[dist <= half]).mean())
    far = float((pred[dist > half] == yte[dist > half]).mean())

    return {"seq_len": L, "n_pairs": P, "vocab": cfg.vocab_size,
            "accuracy": acc, "chance": chance_accuracy(cfg),
            "acc_near": near, "acc_far": far, "steps": step,
            "seconds": time.time() - t0}

## 3. Pha 1 - Do tham: attention giai duoc toi do dai nao?

Chi chay attention (nhanh nhat) de tim tran do dai, truoc khi tieu GPU cho Hyena. Tieu chi dat: do chinh xac > doan mo + 0,30.

In [ ]:
LENGTHS = [33, 65, 129, 257]

scout = []
print(f"{'L':>5}{'cap':>5}{'vocab':>7}{'acc':>8}{'doan mo':>9}{'gan':>7}{'xa':>7}{'giay':>7}")
print("-" * 55)
for sl in LENGTHS:
    r = run_one(sl, "AA")
    r["config"] = "attention"
    ok = r["accuracy"] > r["chance"] + 0.30
    r["solvable"] = ok
    scout.append(r)
    print(f"{r['seq_len']:>5}{r['n_pairs']:>5}{r['vocab']:>7}{r['accuracy']:>8.3f}"
          f"{r['chance']:>9.3f}{r['acc_near']:>7.3f}{r['acc_far']:>7.3f}{r['seconds']:>7.0f}"
          + ("  giai duoc" if ok else "  chua dat"))

solvable = [r for r in scout if r["solvable"]]
if solvable:
    # LAY DO DAI LON NHAT - cang dai cang lo ra khac biet giua cac toan tu.
    # (Notebook truoc lay solvable[-1] theo thu tu danh sach, vo tinh trung cau
    #  hinh DE NHAT, khien ca bon cau hinh deu bao hoa o 1,000.)
    TARGET = max(solvable, key=lambda r: r["seq_len"])
    print(f"\n=> Dung L = {TARGET['seq_len']} cho pha 2 "
          f"({TARGET['n_pairs']} cap, vocab {TARGET['vocab']}, "
          f"attention dat {TARGET['accuracy']:.3f})")
else:
    TARGET = None
    print("\nKHONG do dai nao giai duoc. DUNG LAI, khong chay pha 2.")
    print("Huong xu ly: tang STEPS_TARGET, hoac giam so cap bang cach dung")
    print("vocab lon hon n_pairs de tac vu bot chat.")

pd.DataFrame(scout).to_csv("results/E6_scout_attention.csv", index=False)

## 4. Pha 2 - So sanh bon cau hinh o do dai kho nhat con giai duoc

| Cau hinh | Y nghia |
|---|---|
| `attention` | moc tren, truy cap truc tiep moi vi tri |
| `hyena_uniform` | khoang alpha nhom tu chon (doi chung 1) |
| `hyena_logspace` | do dai hieu dung trai deu theo log (doi chung 2, cong bang) |
| `hyena_corpus` | alpha suy tu corpus (de xuat), trung vi ~3 token |

**Du doan can kiem chung:** `corpus` co do dai hieu dung rat ngan nen se kem nhat, dac biet o cot `acc_far`. Neu dung, do la phat hien co gia tri: khoi tao dua tren du lieu **danh doi kha nang tam xa lay chat luong cuc bo**.

In [ ]:
from hyena_study.morphology import alphas_from_mi, logspaced_alphas

assert TARGET is not None, "Pha 1 khong tim duoc do dai giai duoc - khong chay tiep"
L = TARGET["seq_len"]
D_MODEL = 64

# Alpha phai SINH LAI cho dung d_model va do dai cua tac vu nay. Dung lai file
# alpha sinh cho d_model=256 / L=512 se lam sai do dai hieu dung.
mi = pd.read_csv("results/E0b_mi_decay_vi_bpe_k500.csv")
corpus_alpha = alphas_from_mi(mi["lag"].values, mi["mi_corrected_nats"].values,
                              d_model=D_MODEL, seq_len=L).alpha
logspace_alpha = logspaced_alphas(D_MODEL, seq_len=L).alpha

for nm, a in (("corpus", corpus_alpha), ("logspace", logspace_alpha)):
    e = L / np.array(a)
    print(f"  {nm:<9} do dai hieu dung: trung vi {np.median(e):.1f} token, "
          f"{(e <= 4).sum()}/{D_MODEL} kenh <= 4 token, xa nhat {e.max():.0f}")

CONFIGS = [
    ("attention",      "AA", None),
    ("hyena_uniform",  "HH", None),
    ("hyena_logspace", "HH", logspace_alpha),
    ("hyena_corpus",   "HH", corpus_alpha),
]
SEEDS = [0, 1]

rows = []
print(f"\n{'cau hinh':<16}{'seed':>5}{'acc':>8}{'gan':>7}{'xa':>7}{'giay':>7}")
print("-" * 50)
for name, spec, alpha in CONFIGS:
    for sd in SEEDS:
        r = run_one(L, spec, alpha=alpha, seed=sd, d_model=D_MODEL)
        r["config"] = name; r["seed"] = sd
        rows.append(r)
        print(f"{name:<16}{sd:>5}{r['accuracy']:>8.4f}{r['acc_near']:>7.3f}"
              f"{r['acc_far']:>7.3f}{r['seconds']:>7.0f}")

df = pd.DataFrame(rows)
df.to_csv("results/E6_recall_comparison.csv", index=False)

print("\n" + "=" * 66)
print(df.groupby("config")[["accuracy", "acc_near", "acc_far"]]
        .agg(["mean", "std"]).to_string())
print(f"\nMuc doan mo: {rows[0]['chance']:.4f}   |   L = {L}, "
      f"{rows[0]['n_pairs']} cap, vocab {rows[0]['vocab']}")

print("\nCACH DOC - va cai bay phai tranh:")
print("  * Neu MOI cau hinh deu ~1,000 => lai bao hoa, tac vu qua de o do dai nay.")
print("    KHONG duoc ket luan gi; phai tang do dai roi chay lai.")
print("  * corpus co acc_far THAP hon logspace => khoi tao tu du lieu danh doi")
print("    kha nang tam xa. Day la phat hien chinh can tim.")
print("  * Chenh lech nho hon do lech giua hai seed => nhieu, khong ket luan duoc.")

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/E6_ketqua")
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in Path("results").glob("E6*"):
    shutil.copy(f, out / f.name)
shutil.make_archive("/kaggle/working/E6_ketqua", "zip", out)
print("Da gom:", sorted(p.name for p in out.iterdir()))
print("Tai /kaggle/working/E6_ketqua.zip o panel Output ben phai.")